In [21]:
import os
import pandas as pd
import json
import statsmodels.api as sm
from statsmodels.formula.api import ols
import scipy.stats as stats

## Loading the data

In [29]:
def get_results_from_benchmark_file(filename):
    with open(filename, "r") as file:
        response = json.load(file)
    
    result = response["scores"]

    # I mixed up the prompt type and user prompt when writing to the files
    result["user_prompt"] = response["prompt_type"]
    result["generator"] = response["generator"]
    result["prompt_type"] = response["user_prompt"]
    return result

In [30]:
"""Load benchmark"""
output_file = f"../results/result_eval_benchmark.json"
if os.path.exists(output_file):
    os.remove(output_file)

results = []

for generator_folder in os.listdir(f"../results/generation_only"):
            if generator_folder.startswith("."):
                continue

            for result_file in os.listdir(
                f"../results/generation_only/{generator_folder}"
            ):
                if result_file.startswith("."):
                    continue

                results.append(get_results_from_benchmark_file(
                    f"../results/generation_only/{generator_folder}/{result_file}"
                ))

with open(f"../responses/result_eval_benchmark.json", "w") as benchmark_file:
    json.dump(results, benchmark_file)

benchmark_df = pd.DataFrame(results)
benchmark_df

,answer_relevancy,factual_correctness,bleu,user_prompt,generator,prompt_type
0,0.258086,0.75,0.030162,Wat kun je me vertellen over Rick Kruys?,Ministral-3-14B-Instruct-2512,simple_question
1,0.450801,NaN,0.039195,Wat kun je me vertellen over Gemeente Amsterdam?,Ministral-3-14B-Instruct-2512,simple_question
2,0.338714,NaN,0.024387,Wat kun je me vertellen over Texel?,Ministral-3-14B-Instruct-2512,simple_question
3,0.303059,NaN,0.026227,Wat kun je me vertellen over Bruinvis?,Ministral-3-14B-Instruct-2512,simple_question
4,0.308608,NaN,0.020706,Wat kun je me vertellen over het project Zuida...,Ministral-3-14B-Instruct-2512,context_question
...,...,...,...,...,...,...
259,0.233125,NaN,0.025786,Bruinvis,Qwen3-30B-A3B-Instruct-2507,single_term
260,0.306629,NaN,0.072893,Wat kun je me vertellen over Trekkertrek?,Qwen3-30B-A3B-Instruct-2507,simple_question
261,0.000000,NaN,0.061098,Ik ben een journalist uit Noord-Holland en ik ...,Qwen3-30B-A3B-Instruct-2507,journalist_question
262,0.403136,NaN,0.023542,Wat kun je me vertellen over de Gemeente Amste...,Qwen3-30B-A3B-Instruct-2507,context_question


In [60]:
"""Use this when there is a file containing the results directly."""

with open(f"../results/result_eval.json", 'r') as result_file:
    results = json.load(result_file)
    
results_df = pd.DataFrame(results)
multi_index_results_df = results_df.set_index(['embedder', 'generator', 'distance_metric']).sort_index()
multi_index_results_df

context_utilization  \
embedder                       generator                     distance_metric                        
BM25                           Ministral-3-14B-Instruct-2512 nan                              1.0   
                                                             nan                              1.0   
                                                             nan                              1.0   
                                                             nan                              1.0   
                                                             nan                              1.0   
...                                                                                           ...   
multilingual-e5-large-instruct Qwen3-30B-A3B-Instruct-2507   manhattan                        0.0   
                                                             manhattan                        0.0   
                                                             manhattan                        0.0   
                                                             manhattan                        0.0   
                                                             manhattan                        0.0   

                                                                              answer_relevancy  \
embedder                       generator                     distance_metric                     
BM25                           Ministral-3-14B-Instruct-2512 nan                      0.258085   
                                                             nan                      0.399124   
                                                             nan                      0.000000   
                                                             nan                      0.303063   
                                                             nan                      0.308619   
...                                                                                        ...   
multilingual-e5-large-instruct Qwen3-30B-A3B-Instruct-2507   manhattan                0.257784   
                                                             manhattan                0.719094   
                                                             manhattan                0.811106   
                                                             manhattan                0.789209   
                                                             manhattan                0.353120   

                                                                              faithfulness  \
embedder                       generator                     distance_metric                 
BM25                           Ministral-3-14B-Instruct-2512 nan                  0.000000   
                                                             nan                  0.000000   
                                                             nan                  1.000000   
                                                             nan                       NaN   
                                                             nan                       NaN   
...                                                                                    ...   
multilingual-e5-large-instruct Qwen3-30B-A3B-Instruct-2507   manhattan                 NaN   
                                                             manhattan                 NaN   
                                                             manhattan            0.333333   
                                                             manhattan            0.250000   
                                                             manhattan                 NaN   

                                                                              factual_correctness  \
embedder                       generator                     distance_metric                        
BM25                           Ministral-3-14B-Instruct-2512 nan

In [ ]:
def get_results_from_result_file(filename):
    with open(filename, "r") as file:
        response = json.load(file)
    
    result = response["scores"]
    result["user_prompt"] = response["user_prompt"]
    result["embedder"] = response["embedder"]
    result["distance_metric"] = response["distance_metric"]
    result["generator"] = response["generator"]
    result["prompt_type"] = response["prompt_type"]
    return result

In [ ]:
"""Use this when you want to calculate results based on the contents of the individual files."""

output_file = f"../results/result_eval.json"
if os.path.exists(output_file):
    os.remove(output_file)

results = []

for embedder_folder in os.listdir("../results"):
    if embedder_folder.startswith(".") or embedder_folder == "generation_only":
        continue

    if embedder_folder == "BM25":
        for generator_folder in os.listdir(f"../results/{embedder_folder}"):

            if generator_folder.startswith("."):
                continue

            for result_file in os.listdir(
                f"../results/{embedder_folder}/{generator_folder}"
            ):
                if result_file.startswith("."):
                    continue

                results.append(get_results_from_result_file(f"../results/{embedder_folder}/{generator_folder}/{result_file}"))

    else:
        for generator_folder in os.listdir(f"../results/{embedder_folder}"):
            if generator_folder.startswith("."):
                continue

            for distance_metric in os.listdir(
                f"../results/{embedder_folder}/{generator_folder}"
            ):
                if distance_metric.startswith("."):
                    continue

                for result_file in os.listdir(
                    f"../results/{embedder_folder}/{generator_folder}/{distance_metric}"
                ):
                    if distance_metric.startswith("."):
                        continue

                    results.append(get_results_from_result_file(
                        f"../results/{embedder_folder}/{generator_folder}/{distance_metric}/{result_file}"
                    ))

with open(f"../responses/result_eval.json", "w") as result_file:
    json.dump(results, result_file)

results_df = pd.DataFrame(results)
multi_index_results_df = results_df.set_index(['embedder', 'generator', 'distance_metric']).sort_index()
multi_index_results_df

context_utilization  \
embedder                       generator                     distance_metric                        
BM25                           Ministral-3-14B-Instruct-2512 nan                              1.0   
                                                             nan                              1.0   
                                                             nan                              1.0   
                                                             nan                              1.0   
                                                             nan                              1.0   
...                                                                                           ...   
multilingual-e5-large-instruct Qwen3-30B-A3B-Instruct-2507   manhattan                        0.0   
                                                             manhattan                        0.0   
                                                             manhattan                        0.0   
                                                             manhattan                        0.0   
                                                             manhattan                        0.0   

                                                                              answer_relevancy  \
embedder                       generator                     distance_metric                     
BM25                           Ministral-3-14B-Instruct-2512 nan                      0.258085   
                                                             nan                      0.399173   
                                                             nan                      0.000000   
                                                             nan                      0.303063   
                                                             nan                      0.308619   
...                                                                                        ...   
multilingual-e5-large-instruct Qwen3-30B-A3B-Instruct-2507   manhattan                0.257794   
                                                             manhattan                0.719062   
                                                             manhattan                0.811106   
                                                             manhattan                0.789209   
                                                             manhattan                0.353063   

                                                                              faithfulness  \
embedder                       generator                     distance_metric                 
BM25                           Ministral-3-14B-Instruct-2512 nan                  0.000000   
                                                             nan                  0.000000   
                                                             nan                  1.000000   
                                                             nan                       NaN   
                                                             nan                       NaN   
...                                                                                    ...   
multilingual-e5-large-instruct Qwen3-30B-A3B-Instruct-2507   manhattan                 NaN   
                                                             manhattan                 NaN   
                                                             manhattan            0.333333   
                                                             manhattan            0.250000   
                                                             manhattan                 NaN   

                                                                                                                    user_prompt  \
embedder                       generator                     distance_metric                                                      
BM25

In [61]:
results_df

,context_utilization,answer_relevancy,faithfulness,factual_correctness,bleu,user_prompt,embedder,generator,distance_metric,prompt_type,faithfullness
0,0.0,0.000000,0.000000,NaN,0.026280,Wat kun je me vertellen over Rick Kruys?,bge-m3,Ministral-3-14B-Instruct-2512,cosine_similarity,simple_question,NaN
1,0.0,0.000000,NaN,NaN,0.037720,Wat kun je me vertellen over Gemeente Amsterdam?,bge-m3,Ministral-3-14B-Instruct-2512,cosine_similarity,simple_question,NaN
2,0.0,0.338714,NaN,NaN,0.008521,Wat kun je me vertellen over Texel?,bge-m3,Ministral-3-14B-Instruct-2512,cosine_similarity,simple_question,NaN
3,0.0,0.382990,NaN,NaN,0.009348,Wat kun je me vertellen over Bruinvis?,bge-m3,Ministral-3-14B-Instruct-2512,cosine_similarity,simple_question,NaN
4,0.0,0.672666,0.000000,NaN,0.017912,Wat kun je me vertellen over het project Zuida...,bge-m3,Ministral-3-14B-Instruct-2512,cosine_similarity,context_question,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1843,0.0,0.257784,NaN,NaN,0.074956,Bruinvis,multilingual-e5-large-instruct,Qwen3-30B-A3B-Instruct-2507,manhattan,single_term,NaN
1844,0.0,0.719094,NaN,NaN,0.027185,Wat kun je me vertellen over Trekkertrek?,multilingual-e5-large-instruct,Qwen3-30B-A3B-Instruct-2507,manhattan,simple_question,NaN
1845,0.0,0.811106,0.333333,NaN,0.058166,Ik ben een journalist uit Noord-Holland en ik ...,multilingual-e5-large-instruct,Qwen3-30B-A3B-Instruct-2507,manhattan,journalist_question,NaN
1846,0.0,0.789209,0.250000,NaN,0.021602,Wat kun je me vertellen over de Gemeente Amste...,multilingual-e5-large-instruct,Qwen3-30B-A3B-Instruct-2507,manhattan,context_question,NaN


## Initial results

In [62]:
metrics = ['mean', 'std']

grouped = results_df.groupby(['embedder', 'generator', 'distance_metric'], dropna=False).agg({'context_utilization' : metrics,
                                                                                              'answer_relevancy' : metrics,
                                                                                              'faithfulness' : metrics,
                                                                                              'factual_correctness' : metrics,
                                                                                              'bleu' : metrics})
grouped

context_utilization  \
                                                                                                       mean   
embedder                       generator                              distance_metric                         
BM25                           Ministral-3-14B-Instruct-2512          NaN                          0.847588   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 NaN                          0.619792   
                               Qwen3-30B-A3B-Instruct-2507            NaN                          0.750992   
Qwen3-Embedding-8B             Ministral-3-14B-Instruct-2512          cosine_similarity            0.074612   
                                                                      manhattan                    0.102713   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 cosine_similarity            0.000000   
                                                                      manhattan                    0.019380   
                               Qwen3-30B-A3B-Instruct-2507            cosine_similarity            0.044061   
                                                                      manhattan                    0.055882   
bge-m3                         Ministral-3-14B-Instruct-2512          cosine_similarity            0.105364   
                                                                      manhattan                    0.329502   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 cosine_similarity            0.000000   
                                                                      manhattan                    0.076705   
                               Qwen3-30B-A3B-Instruct-2507            cosine_similarity            0.043103   
                                                                      manhattan                    0.247126   
multilingual-e5-large-instruct Ministral-3-14B-Instruct-2512          cosine_similarity            0.088294   
                                                                      manhattan                    0.225379   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 cosine_similarity            0.000000   
                                                                      manhattan                    0.035441   
                               Qwen3-30B-A3B-Instruct-2507            cosine_similarity            0.045977   
                                                                      manhattan                    0.212644   

                                                                                                   \
                                                                                              std   
embedder                       generator                              distance_metric               
BM25                           Ministral-3-14B-Instruct-2512          NaN                0.287192   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 NaN                0.426332   
                               Qwen3-30B-A3B-Instruct-2507            NaN                0.382318   
Qwen3-Embedding-8B             Ministral-3-14B-Instruct-2512          cosine_similarity  0.243889   
                                                                      manhattan          0.238348   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 cosine_similarity  0.000000   
                                                                      manhattan          0.109719   
                               Qwen3-30B-A3B-Instruct-2507            cosine_similarity  0.202471   
                                                                      manhattan          0.182478   
bge-m3                         Ministral-3-14B-Instruct-2512          cosine_similarity  0.295438   
                                                                      manhattan          0.382285   
         

## ANOVA

In [63]:
# Handle missing values in categorical variables (e.g., BM25 doesn't have a distance metric)
results_df['distance_metric'] = results_df['distance_metric'].fillna('None')

metrics = ['answer_relevancy', 'context_utilization', 'factual_correctness', 'bleu']

for metric in metrics:
    # Handle missing values in your target variable if necessary
    anova_df = results_df.dropna(subset=[metric])

    # Define and fit the Ordinary Least Squares (OLS) model
    # Wrap categorical variables in C() to tell statsmodels they are factors
    formula = f'{metric} ~ C(embedder) + C(generator) + C(distance_metric) + C(prompt_type)'
    model = ols(formula, data=anova_df).fit()

    # Perform a Type II ANOVA (recommended for unbalanced/unequal group sizes)
    anova_table = sm.stats.anova_lm(model, typ=2)

    print(f'ANOVA of {metric}')
    
    # Display the results
    print(anova_table)
    print()

ANOVA of answer_relevancy
                        sum_sq      df          F        PR(>F)
C(embedder)           0.010072     3.0   0.048257  8.261503e-01
C(generator)          8.357171     2.0  60.060001  5.642841e-26
C(distance_metric)    0.006715     2.0   0.048257  8.261503e-01
C(prompt_type)        4.889596     3.0  23.426522  7.132731e-15
Residual            124.954031  1796.0        NaN           NaN

ANOVA of context_utilization
                        sum_sq      df          F        PR(>F)
C(embedder)           0.105809     3.0   0.487140  4.852988e-01
C(generator)          6.283206     2.0  43.391329  4.055030e-19
C(distance_metric)    0.070539     2.0   0.487140  4.852988e-01
C(prompt_type)        0.404285     3.0   1.861308  1.341270e-01
Residual            126.630456  1749.0        NaN           NaN

ANOVA of factual_correctness
                      sum_sq    df         F    PR(>F)
C(embedder)         0.193988   3.0  0.444243  0.722876
C(generator)        0.185003   2.0  

/Users/maritvandenhelder/miniconda3/envs/thesis/lib/python3.14/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 3, but rank is 1
  warnings.warn('covariance of constraints does not have full '
/Users/maritvandenhelder/miniconda3/envs/thesis/lib/python3.14/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 2, but rank is 1
  warnings.warn('covariance of constraints does not have full '
/Users/maritvandenhelder/miniconda3/envs/thesis/lib/python3.14/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 3, but rank is 1
  warnings.warn('covariance of constraints does not have full '
/Users/maritvandenhelder/miniconda3/envs/thesis/lib/python3.14/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints

## Paired t-test

In [69]:
# Merge on the prompt and generator
merged_df = pd.merge(
    benchmark_df[['user_prompt', 'answer_relevancy', 'generator','factual_correctness', 'bleu']], 
    results_df[['embedder', 'generator', 'distance_metric', 'user_prompt', 'answer_relevancy', 'factual_correctness', 'bleu']], 
    on=['user_prompt', 'generator'], 
    suffixes=('_baseline', '_pipeline')
).dropna()  
merged_df['answer_relevancy_difference'] = merged_df['answer_relevancy_pipeline'] - merged_df['answer_relevancy_baseline']
merged_df['factual_correctness_difference'] = merged_df['factual_correctness_pipeline'] - merged_df['factual_correctness_baseline']
merged_df['bleu_difference'] = merged_df['bleu_pipeline'] - merged_df['bleu_baseline']

merged_df

,user_prompt,answer_relevancy_baseline,generator,factual_correctness_baseline,bleu_baseline,embedder,distance_metric,answer_relevancy_pipeline,factual_correctness_pipeline,bleu_pipeline,answer_relevancy_difference,factual_correctness_difference,bleu_difference
4,Wat kun je me vertellen over Rick Kruys?,0.258086,Ministral-3-14B-Instruct-2512,0.75,0.030162,BM25,None,0.258085,0.50,0.031760,-8.097529e-07,-0.25,0.001598
133,Wat kun je me vertellen over Haarlem?,0.416903,Ministral-3-14B-Instruct-2512,0.50,0.046994,bge-m3,cosine_similarity,0.416915,0.00,0.018180,1.232513e-05,-0.50,-0.028815
134,Wat kun je me vertellen over Haarlem?,0.416903,Ministral-3-14B-Instruct-2512,0.50,0.046994,bge-m3,manhattan,0.416915,0.00,0.006888,1.232513e-05,-0.50,-0.040107
137,Wat kun je me vertellen over Haarlem?,0.416903,Ministral-3-14B-Instruct-2512,0.50,0.046994,BM25,None,0.000000,0.80,0.017224,-4.169028e-01,0.30,-0.029770
199,Amsterdam,0.351868,Ministral-3-14B-Instruct-2512,0.40,0.012945,Qwen3-Embedding-8B,manhattan,0.000000,1.00,0.005881,-3.518683e-01,0.60,-0.007063
200,Amsterdam,0.351868,Ministral-3-14B-Instruct-2512,0.40,0.012945,BM25,None,0.351868,0.00,0.000000,0.000000e+00,-0.40,-0.012945
354,Wat kun je me vertellen over Fotofestivals in ...,0.314865,Ministral-3-14B-Instruct-2512,0.00,0.024427,BM25,None,0.000000,0.00,0.049170,-3.148646e-01,0.00,0.024743
400,Rick Kruys,0.202499,Ministral-3-14B-Instruct-2512,0.00,0.016961,bge-m3,manhattan,0.000000,0.00,0.000000,-2.024994e-01,0.00,-0.016961
401,Rick Kruys,0.202499,Ministral-3-14B-Instruct-2512,0.00,0.016961,Qwen3-Embedding-8B,cosine_similarity,0.811301,0.00,0.000000,6.088014e-01,0.00,-0.016961
402,Rick Kruys,0.202499,Ministral-3-14B-Instruct-2512,0.00,0.016961,Qwen3-Embedding-8B,manhattan,0.225628,1.00,0.000000,2.312862e-02,1.00,-0.016961


In [70]:
# Define a custom function to calculate the p-value
def p_value(series):
    # Drop NaNs to prevent errors in the t-test
    clean_series = series.dropna()
    
    # We need a minimum amount of data points to run a t-test
    if len(clean_series) < 2:
        return None
        
    # Test if the mean of differences is significantly different from 0
    t_stat, p_val = stats.ttest_1samp(clean_series, popmean=0)
    return p_val

metrics = ['mean', 'std', p_value]

merged_grouped = merged_df.groupby(['embedder', 'generator', 'distance_metric'], dropna=False).agg({'answer_relevancy_difference' : metrics,
                                                                                                    'factual_correctness_difference' : metrics,
                                                                                                    'bleu_difference' : metrics})
merged_grouped

answer_relevancy_difference  \
                                                                                                               mean   
embedder                       generator                              distance_metric                                 
BM25                           Ministral-3-14B-Instruct-2512          None                            -1.219614e-01   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 None                             6.949392e-03   
                               Qwen3-30B-A3B-Instruct-2507            None                            -1.339905e-01   
Qwen3-Embedding-8B             Ministral-3-14B-Instruct-2512          cosine_similarity                6.088014e-01   
                                                                      manhattan                       -1.643698e-01   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 manhattan                        1.335066e-01   
                               Qwen3-30B-A3B-Instruct-2507            manhattan                        1.110223e-16   
bge-m3                         Ministral-3-14B-Instruct-2512          cosine_similarity                1.232513e-05   
                                                                      manhattan                       -1.012435e-01   
                               Qwen3-30B-A3B-Instruct-2507            cosine_similarity                1.369432e-01   
multilingual-e5-large-instruct Ministral-3-14B-Instruct-2512          cosine_similarity                3.519001e-01   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 manhattan                        2.244224e-01   
                               Qwen3-30B-A3B-Instruct-2507            cosine_similarity                0.000000e+00   

                                                                                                   \
                                                                                              std   
embedder                       generator                              distance_metric               
BM25                           Ministral-3-14B-Instruct-2512          None               0.191677   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 None               0.009825   
                               Qwen3-30B-A3B-Instruct-2507            None               0.232089   
Qwen3-Embedding-8B             Ministral-3-14B-Instruct-2512          cosine_similarity       NaN   
                                                                      manhattan          0.265163   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 manhattan          0.614907   
                               Qwen3-30B-A3B-Instruct-2507            manhattan               NaN   
bge-m3                         Ministral-3-14B-Instruct-2512          cosine_similarity       NaN   
                                                                      manhattan          0.143197   
                               Qwen3-30B-A3B-Instruct-2507            cosine_similarity       NaN   
multilingual-e5-large-instruct Ministral-3-14B-Instruct-2512          cosine_similarity       NaN   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 manhattan               NaN   
                               Qwen3-30B-A3B-Instruct-2507            cosine_similarity       NaN   

                                                                                                   \
                                                                                          p_value   
embedder                       generator                              distance_metric               
BM25                           Ministral-3-14B-Instruct-2512          None               0.179836   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 None               0.499899   
                               Qwen3-30B-A3B-Instruct-2507   